In [2]:

'''
core MIP entries, generally singular, and do not change often. 
'''
import json 

repo_ctx = json.load(open('../context.json'))
# repo_ctx['@depends'] = ['mip-cmor-tables:auxillary/mip_era',]

activity=[] 

from jsonschema import validate,exceptions

'''
cmip6plus/experiment
'''

def validate_json(data, schema, name = 'none'):
    try:
        validate(instance=data, schema=schema)
        print(f"Validation succeeded: {name}")
    except exceptions.ValidationError as err:
        print("Validation error:", err.message)

In [3]:
schema = {}

loc = f'./id/schema.json'
json.dump(schema, open(loc,'w'),indent=4)

In [11]:
experiment = json.load(open('../../CMIP6Plus_experiment_id.json'))['experiment_id']
list(experiment.values())[0]

{'activity_id': ['CMIP'],
 'additional_allowed_model_components': ['AER', 'CHEM', 'BGC'],
 'description': 'DECK: 1pctCO2',
 'end': '',
 'experiment': '1 percent per year increase in CO2',
 'experiment_id': '1pctCO2',
 'min_number_yrs_per_sim': 150,
 'parent_activity_id': ['CMIP'],
 'parent_experiment_id': ['piControl'],
 'required_model_components': ['AOGCM'],
 'start': '',
 'sub_experiment_id': ['none'],
 'tier': 1}

In [20]:


for exe in experiment.values():

    eid = exe['experiment_id']
    data = {
        "@id":"cmip6plus:experiment/id/"+eid.lower(),
        "@type":"cmip:experiment_id",
        "experiment_id:experiment_id": exe['experiment_id'],
        "experiment_id:experiment": exe['experiment'],
        "experiment_id:description": exe['description'],
        "experiment_id:activity_id":{"@set":[{"@id":f'cmip6plus:activity/{i.lower()}'} for i in exe["activity_id"]]},
        "experiment_id:model_components":{"@nest":{
            "experiment_id:required":{"@set":[{"@id":f'cmip6plus:source/type/{i.lower()}'} for i in exe['required_model_components']]},
            "experiment_id:additional_allowed":{"@set":[{"@id":f'cmip6plus:source/type/{i.lower()}'} for i in exe['additional_allowed_model_components']]},
        },
        "experiment_id:parameters":{"@nest":{
            "experiment_id:start": int(exe['start'] or -999),
            "experiment_id:end": int(exe['end'] or -999),
            "experiment_id:'min_number_yrs_per_sim'": exe['min_number_yrs_per_sim'],
        },},
        "experiment_id:parent":{"@nest":{
            "experiment_id:activity_id":{"@set":[{"@id":f'cmip6plus:activity/{i.lower()}'} for i in exe["parent_activity_id"]]},
            "experiment_id:experiment_id":{"@set":[{"@id":f'cmip6:experiment/id/{i.lower()}'} for i in exe["parent_experiment_id"]]}
        },},
        'experiment_id:sub_experiment_id': {"@id":f"cmip6plus:experiment/sub_id/{exe['sub_experiment_id'][0]}"},
        'experiment_id:tier': int(exe['tier'] or -999)}
    }
        
        
        

    # validate_json(data,schema,eid)

    loc = f'./id/{eid.lower()}.json'
    json.dump(data, open(loc,'w'),indent=4)
        
        

In [7]:
ctx = repo_ctx['@context'].copy()
ctx.update({"@vocab":"source_id:",
            "@depends":['mip-cmor-tables:license','cmip6plus:model/components','cmip6plus_cvs:activity','cmip6plus_cvs:source/type'],
            "license":{"@vocab":'license'}})

loc = f'./id/context.json'
json.dump(ctx, open(loc,'w'),indent=4)

In [22]:
seid = json.load(open('../../CMIP6Plus_sub_experiment_id.json'))['sub_experiment_id']


for sid,desc in seid.items():

    data = {
        "@id":"cmip6plus:experiment/sub_id/"+sid.lower(),
        "@type":"cmip:sub_experiment_id",
        "sub_experiment_id:sub_experiment_id": sid,
        "sub_experiment_id:description": desc,

    }
        
    
    # validate_json(data,schema,eid)

    loc = f'./sub_id/{sid.lower()}.json'
    json.dump(data, open(loc,'w'),indent=4)
        
        


In [9]:
ctx = repo_ctx['@context'].copy()
ctx.update({"@vocab":"source_type:"})

loc = f'./type/context.json'
json.dump(ctx, open(loc,'w'),indent=4)